In [4]:
import time

from faker import Faker
import mysql.connector
import random

from mysql.connector import MySQLConnection

In [5]:
def get_engine(config) -> mysql.connector.connection.MySQLConnection:
    return mysql.connector.connect(**config)


def insert_loop(connection: MySQLConnection, faker: Faker, n: int):
    cursor = connection.cursor()
    customer_sql = "INSERT INTO customers (name, email, address, phone_number, company) VALUES (%s, %s, %s, %s, %s)"
    employee_sql = "INSERT INTO employees (name, department, salary) VALUES (%s, %s, %s)"
    department_sql = "INSERT INTO departments (name) VALUES (%s)"

    try:
        connection.start_transaction()
        for _ in range(n):
            customer_values = (faker.name(), faker.unique.email(), faker.address(), faker.phone_number(), faker.company())
            cursor.execute(customer_sql, customer_values)
            employee_values = (faker.name(), random.randint(1, 10), round(random.uniform(3000, 12000), 2))
            cursor.execute(employee_sql, employee_values)
            department_values = (faker.job()[:50],)
            cursor.execute(department_sql, department_values)
            connection.commit()
            time.sleep(0.1)
        print(f"✅ customers/employees 각각 {n}건 INSERT 완료")
    except Exception as e:
        connection.rollback()
        print("❌ rollback:", e)
    finally:
        cursor.close()
        connection.close()

In [6]:
insert_loop(get_engine({"host": "mysql-primary", "port": 3306, "user": "root", "password": "root", "database": "mmix", }), Faker("ko_KR"), 5)

✅ customers/employees 각각 5건 INSERT 완료
